# 03. Modelling: from the mean baseline to the tuned CatBoost

**Project:** Kaggle House Prices. **Stage:** after feature engineering (notebook 02), before ensembles (notebook 04).

This notebook is thin: it reads the experiment registry (`experiments/registry.jsonl`, read-only), draws the model ladder, reproduces one registered experiment live with `houseprices.evaluate`, and plots the Optuna trial histories saved by the tuning script. It never writes the registry, never fits on all rows and never touches the holdout.

All scores are RMSE on log1p(SalePrice), KFold(5, shuffle, seed 42) on the 1,241 development rows, lower is better. They are **selection-fold scores**: the best of many candidates is optimistic by about 0.002 to 0.003, and differences below 0.003 are not wins (see `docs/drafts/STUDY_GUIDE_part2a_validation.md`).

In [ ]:
# Resolve the project root the same way notebook 01 does: walk up until data_description.txt is found.
import os
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "data_description.txt").exists():
    if root == root.parent:
        raise FileNotFoundError("Project root (data_description.txt) not found")
    root = root.parent
os.chdir(root)
print("Working directory:", Path.cwd())

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from houseprices import evaluate, registry

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 90)
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. The model ladder

Each step is one registered experiment. Each rung changes one thing: E001 is the mean, E003 a Ridge on one-hot features, E041 adds the engineered features and the skew correction, E056 is the tuned Lasso, E060 the Stage 1 CatBoost (25 Optuna trials) and E066 the Stage 2 CatBoost (75 trials), the final single model.

In [ ]:
reg = registry.load_registry()
ladder_ids = ["E001", "E003", "E041", "E056", "E060", "E066"]
ladder = reg[reg["id"].isin(ladder_ids)].set_index("id").loc[ladder_ids]
table = ladder[["title", "cv_mean", "cv_std"]].copy()
table["step vs previous"] = table["cv_mean"].diff().round(5)
display(table.round(5))
print("Registry records:", len(reg), "| latest id:", reg["id"].iloc[-1])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#9e9e9e", "#4c78a8", "#4c78a8", "#4c78a8", "#e45756", "#e45756"]
ax.bar(ladder.index, ladder["cv_mean"], yerr=ladder["cv_std"] / np.sqrt(5), color=colors, capsize=3)
for i, v in enumerate(ladder["cv_mean"]):
    ax.text(i, v + 0.012, f"{v:.4f}", ha="center", fontsize=9)
ax.set_ylabel("CV RMSE on log1p(SalePrice)")
ax.set_title("Model ladder (error bars: fold std / sqrt(5))")
plt.tight_layout()
plt.show()

The mean baseline scores 0.399. Regularised linear models bring it near 0.13, the tuned Lasso reaches about 0.125, and the tuned boosters go below 0.12. The last step (E060 to E066, 0.0014) is inside the 0.003 noise band: it is not a win by our rule.

## 2. Reproduce a registered experiment live

`evaluate.run_registered` rebuilds the estimator from the registry (`builder` and `builder_kwargs`), loads the same development data and runs the same seeded folds. E056 (Lasso) takes about 2 seconds, so we run it here as a demonstration that the registered number is reproducible.

In [ ]:
report = evaluate.run_registered("E056", verbose=False)
print({k: report[k] for k in ["experiment", "title", "cv", "registered_fold_scores", "reproduced_fold_scores", "reproduced_mean", "fit_seconds"]})
print("Reproduces the registered scores:", report["reproduces_registered_scores"])

## 3. Optuna trial histories (Stage 1 = trials 0 to 24, Stage 2 = trials 25 to 74)

The per-trial CSVs are written by `experiments/tuning/run_optuna_stage1.py`. XGBoost has only 25 trials (no Stage 2, ruling D19). The dashed line marks the end of Stage 1; the red line is the best-so-far score.

In [ ]:
fs = "base+areas+baths+ages+quality_interactions+ordinal_quality"
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, name in zip(axes, ["xgboost", "lightgbm", "catboost"]):
    df = pd.read_csv(f"experiments/tuning/stage1_{name}_{fs}_trials.csv")
    df = df[df["state"] == "COMPLETE"].sort_values("number")
    v = df["value"].to_numpy()
    ax.scatter(df["number"], v, s=14, alpha=0.6, color="#4c78a8")
    ax.plot(df["number"], np.minimum.accumulate(v), color="#e45756", lw=2)
    if len(df) > 25:
        ax.axvline(24.5, color="k", ls="--", lw=1)
    ax.set_ylim(v.min() - 0.002, np.percentile(v, 95) + 0.004)
    ax.set_title(f"{name}: {len(df)} trials, best {v.min():.5f}")
    ax.set_xlabel("trial number")
axes[0].set_ylabel("CV RMSE (mean of 5 folds)")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for name in ["xgboost", "lightgbm", "catboost"]:
    df = pd.read_csv(f"experiments/tuning/stage1_{name}_{fs}_trials.csv")
    df = df[df["state"] == "COMPLETE"].sort_values("number")
    v = df["value"].to_numpy()
    rows.append({"booster": name, "trials": len(v), "best@25": v[:25].min(), "best@75": v.min(),
                 "std first 25": v[:25].std(ddof=1), "std last 25": v[-25:].std(ddof=1) if len(v) > 25 else np.nan})
display(pd.DataFrame(rows).round(5))

TPE concentrates its trials after the first 10 random ones, so the spread of the scores shrinks, and both boosters stopped improving about 30 trials before the end. Both best parameter sets touch an edge of the search space (2,000 trees, `min_child_samples` 5 for LightGBM); we judged the possible gain to be below the 0.003 noise band and closed tuning (ruling D21). The details are in `docs/drafts/STUDY_GUIDE_part2c_boosting_optuna.md`.